# 03 - Curate each GSE and save curated h5ad

Set the standard obs/var schema **explicitly** here (you are the curator). Original obs columns are kept. Save to `data/curated_h5ad/{dataset_id}.h5ad`.

Required obs columns (unknown is OK):
`cell_id_original, cell_id, sample_id, sample_label, source_accession, parent_gse, dataset_id, species, disease_area, tissue, region, assay, technology, enrichment, disease_status, disease_model, genotype, treatment, age, age_month, sex, replicate, data_status, processing_status, source_file`.

Required var columns: `gene_id, gene_symbol, gene_symbol_upper, ensembl_id, feature_type`.

In [ ]:
import sys
from pathlib import Path

# locate the v2 project root (folder containing config/dataset_manifest.yaml)
ROOT = Path.cwd()
while not (ROOT / "config" / "dataset_manifest.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
print("project root:", ROOT)

import manifest_utils as mf
man = mf.load_manifest()
paths = mf.project_paths(ROOT)
mf.ensure_dirs(paths)

import anndata_utils as au

adatas = au.load_h5ad_collection(paths['interim'])
list(adatas.keys())

## Explicit example: GSE287569

This shows the fully manual style. You can set every column by hand.

In [ ]:
acc = "GSE287569"
ds = mf.get_dataset(man, acc)
adata = adatas[acc]

# --- manual obs assignment (edit freely) ---
adata.obs["dataset_id"] = ds["dataset_id"]
adata.obs["source_accession"] = "GSE287569"
adata.obs["parent_gse"] = "GSE287569"
adata.obs["species"] = "Mus musculus"
adata.obs["disease_area"] = "ALS"
adata.obs["tissue"] = "spinal cord"
adata.obs["region"] = "spinal cord"
adata.obs["assay"] = "scRNA-seq"
adata.obs["technology"] = "10x H5"
adata.obs["enrichment"] = "whole spinal cord-derived preparation"
adata.obs["disease_model"] = "SOD1G93A"
adata.obs["data_status"] = ds["data_status"]  # raw_or_filtered_count
# genotype / treatment vary per sample -> inspect sample_label and set, e.g.:
# import numpy as np
# lab = adata.obs["sample_label"].str.lower()
# adata.obs.loc[lab.str.contains("wt"), "genotype"] = "WT"
# adata.obs.loc[lab.str.contains("sod1"), "genotype"] = "SOD1G93A"

au.ensure_standard_obs_columns(adata)   # fill the rest with unknown
au.ensure_standard_var_columns(adata)
au.make_obs_names_unique(adata, prefix="GSE287569")
au.save_h5ad(adata, paths["curated"] / (ds["dataset_id"] + ".h5ad"), overwrite=True)
adata.obs.head()

## Convenience curator for the remaining datasets

Pulls the reference `metadata` block from the manifest, sets `data_status` from the manifest, fills the rest with `unknown`, and keeps original obs columns. Override anything per dataset as needed.

In [ ]:
def curate_from_manifest(adata, ds, **overrides):
    meta = dict(ds.get('metadata', {}))
    adata.obs['dataset_id'] = ds['dataset_id']
    adata.obs['source_accession'] = ds['source_accession']
    adata.obs['parent_gse'] = ds.get('parent_gse', ds['source_accession'])
    adata.obs['data_status'] = ds.get('data_status', 'unknown')
    for k, v in meta.items():
        adata.obs[k] = v
    for k, v in overrides.items():
        adata.obs[k] = v
    au.ensure_standard_obs_columns(adata)
    au.ensure_standard_var_columns(adata)
    au.make_obs_names_unique(adata, prefix=ds['source_accession'])
    return adata

In [ ]:
# curate + save every dataset that is currently loaded
for acc, adata in adatas.items():
    ds = mf.get_dataset(man, acc)
    curate_from_manifest(adata, ds)
    au.save_h5ad(adata, paths['curated'] / (ds['dataset_id'] + '.h5ad'), overwrite=True)
print('curated:', sorted(p.name for p in paths['curated'].glob('*.h5ad')))

> Reminder: TPM (GSE167331) stays `processed_TPM`, SoupX (GSE206330) stays `processed_SoupX_corrected`, RDS (GSE295514) stays `RDS_converted_unknown_or_counts`. These are taken from the manifest `data_status` and must not be relabelled as raw counts.

Next: **04_merge_curated_h5ad.ipynb**